In [1]:
from src.parsers.mineru_parser import MinerUParser
from src.extractors.cybersec_extractor import CybersecEntityExtractor
from src.knowledge_graph.neo4j_manager import Neo4jManager
from src.retrieval.hybrid_retrieval import HybridRetriever

c:\Users\ishani.kathuria\projects\TrustworthyRAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# # Parse document
# parser_config = {"dtype": "auto", "device_map": "auto"}
# parser = MinerUParser(parser_config)
# parsed_doc = parser.parse("data/WickedRose_andNCPH.pdf")

In [3]:
# extractor_config = {
#     "spacy_model": "en_core_web_lg"
# }
# extractor = CybersecEntityExtractor(extractor_config)

# # Extract from your parsed document
# entities = extractor.extract_entities(parsed_doc.text)
# relations = extractor.extract_relations(parsed_doc.text, entities)

In [4]:
neo4j_manager = Neo4jManager(
    uri="neo4j://127.0.0.1:7687",
    username="neo4j",
    password="test1234"
)

2025-10-29 14:39:27,245 - Neo4jManager - INFO - Connected to Neo4j at neo4j://127.0.0.1:7687
2025-10-29 14:39:27,272 - Neo4jManager - INFO - Neo4j schema initialized


In [5]:
# entity_stats = neo4j_manager.ingest_entities(entities)
# relation_stats = neo4j_manager.ingest_relations(relations)
# stats = neo4j_manager.get_statistics()
# print(f"Knowledge Graph Stats: {stats}")

In [6]:
malware_query = """
MATCH (m:MALWARE)
RETURN m.text as name, m.confidence as confidence
ORDER BY m.confidence DESC
LIMIT 10
"""
malware = neo4j_manager.query_graph(malware_query)
print(f"\nTop Malware: {malware}")

# Close connection
neo4j_manager.close()

2025-10-29 14:39:27,312 - Neo4jManager - INFO - Neo4j connection closed



Top Malware: [{'name': 'GinWui', 'confidence': 0.9}, {'name': 'GinWui', 'confidence': 0.9}, {'name': 'GinWui', 'confidence': 0.9}, {'name': 'GinWui', 'confidence': 0.9}, {'name': 'GinWui', 'confidence': 0.9}, {'name': 'GinWui', 'confidence': 0.9}, {'name': 'GinWui', 'confidence': 0.9}, {'name': 'GinWui', 'confidence': 0.9}, {'name': 'GinWui', 'confidence': 0.9}, {'name': 'rootkit', 'confidence': 0.9}]


In [7]:
# Ollama configuration
retriever_config = {
    'model_name': 'llama3.1:8b',
    'embedding_model': 'nomic-embed-text',
    'ollama_base_url': 'http://localhost:11434'
}

retriever = HybridRetriever(neo4j_manager, retriever_config)

2025-10-29 14:39:29,671 - HybridRetriever - WARNING - Could not initialize Cypher chain: 1 validation error for GraphCypherQAChain
graph
  Input should be an instance of GraphStore [type=is_instance_of, input_value=<langchain_neo4j.graphs.n...t at 0x000001F903D46D80>, input_type=Neo4jGraph]
    For further information visit https://errors.pydantic.dev/2.11/v/is_instance_of
2025-10-29 14:39:32,554 - HybridRetriever - INFO - Vector store initialized with Ollama embeddings
2025-10-29 14:39:32,556 - HybridRetriever - INFO - LangChain components initialized with Ollama model: llama3.1:8b


In [8]:
# Ask questions
questions = [
    "What malware is mentioned in the documents?",
    "Who created GinWui and what is it?",
    "What organizations are involved in these attacks?",
    "Describe the attack timeline"
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")
    answer = retriever.answer_question(question, method="graph")
    print(f"A: {answer}\n")


Q: What malware is mentioned in the documents?
A: The malware mentioned in the documents are:

1. Rootkit
2. GinWui


Q: Who created GinWui and what is it?
A: Based on the provided knowledge graph, here is the answer:

GinWui is a malware. The relationship between GinWui and "Chinese" is that it exploits something related to Chinese entities, but there is no explicit indication of who created GinWui.


Q: What organizations are involved in these attacks?
A: Based on the provided knowledge graph, the following organizations are involved in the attacks:

1. Wicked Rose
2. Microsoft
3. CHM (likely referring to a specific organization or entity, but its full name is not specified)
4. NCPH (likely referring to a specific organization or entity, but its full name is not specified)

Note that Sichuan University of Science & Engineering is mentioned in the context, but it does not have any relationship with the attacks as per the provided information.


Q: Describe the attack timeline
A: Base